In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import OneHotEncoder

import sys
sys.path.append('../..')
from mount_drive import mount_s_drive

In [2]:
mount_s_drive(subfolder='LCICM/Databases')

Password for ikarhul1: ········


Mounting S drive subfolder LCICM/Databases
The following files and folders have been mounted to /home/idies/workspace/SAFE/:
  mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0
  mimic-iv-1.0
  eICU
  mimic-iv-ecg-parquet
  Kevin
  cardiac_arrest_nantes
  ACCMPMAP
  archive.zip
  Kevin.zip
  mimic-iv-2.2.zip
  mimic-iv-ed-2.2
  Untitled.ipynb
  Simon
  mimic3wdb-matched
  Sampath
  MIMICIII
  MOVER
  MIMICIV
  ENGAGES
  AUMCDB
  mimic-iv-2.2


mkdir: cannot create directory '/home/idies/workspace/SAFE/': File exists


In [3]:
myHours = 60*6

In [4]:
database_folder = '/home/idies/workspace/SAFE/'

### Patients 

In [5]:
ca_ed_df = pd.read_csv('CA_ED.csv')
ca_time_df = pd.read_csv('CA_time.csv')

In [6]:
ids_df1 = ca_ed_df[['subject_id','hadm_id']]
ids_df2 = ca_time_df[['subject_id','hadm_id']]
myIds = pd.concat([ids_df1, ids_df2], ignore_index=True)
myIds = myIds.drop_duplicates()

### myPredictorsDf

In [7]:
ca_ed_df['time'] = ca_ed_df['intime']
ca_time_df['time'] = ca_time_df['max_time']

In [8]:
ca_df = pd.concat([ca_ed_df[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag', 'time']],
                   ca_time_df[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag', 'time']]])
ca_df = ca_df.sort_values(['subject_id','hadm_id'])
ca_df = ca_df.drop_duplicates(['subject_id','hadm_id']).reset_index(drop=True)
duplicate_subjects = ca_df.groupby('subject_id')['hadm_id'].nunique()
duplicate_subjects = duplicate_subjects[duplicate_subjects > 1].index
ca_df = ca_df[~ca_df['subject_id'].isin(duplicate_subjects)]

In [ ]:
ca_df

In [434]:
myPredictorsDf = ca_df[['subject_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag']]
myPredictorsDf['gender'] = (myPredictorsDf['gender'] == 'M').astype(int)
myPredictorsDf.rename(columns={'anchor_age': 'age', 'long_title': 'diagnosis', 'hospital_expire_flag': 'death_at_disch'}, inplace=True)
myPredictorsDf = myPredictorsDf.reset_index(drop=True)

/tmp/ipykernel_2720/1200390782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myPredictorsDf['gender'] = (myPredictorsDf['gender'] == 'M').astype(int)
/tmp/ipykernel_2720/1200390782.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myPredictorsDf.rename(columns={'anchor_age': 'age', 'long_title': 'diagnosis', 'hospital_expire_flag': 'death_at_disch'}, inplace=True)


In [435]:
myPredictorsDf.head()

,subject_id,gender,age,diagnosis,death_at_disch
0,10004720,1,61,NaN,1
1,10010471,0,89,Cardiac arrest due to underlying cardiac condi...,1
2,10038688,1,46,NaN,1
3,10056037,0,68,NaN,1
4,10067389,1,76,Cardiac arrest due to other underlying condition,1


### Extraction Functions

In [436]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['subject_id', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [437]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    subject_ids = aDfToMerge[['subject_id']].copy()
    myPredictorsDf1 = aDfToMerge.copy()
    
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    begin_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', aValueColumn]], on='subject_id', how='left')
        begin_cols[aPrefix + '_first_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    begin_df = pd.concat([subject_ids, pd.DataFrame(begin_cols)], axis=1)
    print()

    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    end_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', aValueColumn]], on='subject_id', how='left')
        end_cols[aPrefix + '_last_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    end_df = pd.concat([subject_ids, pd.DataFrame(end_cols)], axis=1)
    print()
    
    print('Running Agg Columns')
    total = int(aAgg[aTypeColumn].nunique() / 10)
    agg_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', 'max', 'min', 'mean']], on='subject_id', how='left')
        agg_cols.update({aPrefix + '_max_' + value: merged['max'],
                         aPrefix + '_min_' + value: merged['min'],
                         aPrefix + '_mean_' + value: merged['mean']})
        if (i % total == 0):
            print('#', end="")
        i+= 1
    agg_df = pd.concat([subject_ids, pd.DataFrame(agg_cols)], axis=1)
    print()

    myPredictorsDf1 = myPredictorsDf1.merge(begin_df, on='subject_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(end_df, on='subject_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(agg_df, on='subject_id', how='left')
    
    return myPredictorsDf1

In [438]:
def read_by_chunks(file_path, myIds, chunk_size=1e6):
    df_chunks = []
    num_chunks = 0    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size):
        chunk = chunk[chunk['subject_id'].isin(myIds['subject_id'])]
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')    
    return pd.concat(df_chunks, ignore_index=True)

### Chartevents

In [439]:
d_items_df = pd.read_csv(database_folder+'mimic-iv-2.2/icu/d_items.csv')
d_items_df['label'] = d_items_df['label'].str.lower().str.replace(' ', '_')

In [440]:
chartevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/chartevents.csv', myIds)

Chunk 0 Processed.
Chunk 20 Processed.
Chunk 40 Processed.
Chunk 60 Processed.
Chunk 80 Processed.
Chunk 100 Processed.
Chunk 120 Processed.
Chunk 140 Processed.
Chunk 160 Processed.
Chunk 180 Processed.
Chunk 200 Processed.
Chunk 220 Processed.
Chunk 240 Processed.
Chunk 260 Processed.
Chunk 280 Processed.
Chunk 300 Processed.
Processing finished.


In [441]:
chartevents_df = pd.merge(chartevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
chartevents_df['charttime'] = pd.to_datetime(chartevents_df['charttime'],errors='coerce')
chartevents_df['time'] = pd.to_datetime(chartevents_df['time'],errors='coerce')
chartevents_df['chartoffset'] = (chartevents_df['charttime']-chartevents_df['time']).dt.total_seconds()/60
chartevents_df = chartevents_df[chartevents_df['chartoffset']>=0]
chartevents_df = chartevents_df.sort_values(by=['subject_id','chartoffset'])
chartevents_df = chartevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [442]:
columns_to_keep = ['subject_id','value','chartoffset','itemid','label','param_type']
chart_df = chartevents_df[columns_to_keep]
chart_df = chart_df.dropna(subset='label')
chart_df = chart_df[chart_df['chartoffset']<=myHours]

In [443]:
num_df = chart_df[chart_df['param_type']=='Numeric']
labs_df = chart_df[chart_df['param_type']=='Numeric with tag']
bin_df = chart_df[chart_df['param_type']=='Checkbox']
cat_df = chart_df[chart_df['param_type']=='Text']

In [444]:
num_df = num_df.drop(columns=['param_type'])
labs_df = labs_df.drop(columns=['param_type'])
bin_df = bin_df.drop(columns=['param_type'])
cat_df = cat_df.drop(columns=['param_type'])

#### Num

In [445]:
num_df['value'] = pd.to_numeric(num_df['value'], errors='coerce')
num_df = num_df.dropna(subset=['value'])
num_df['itemid'].nunique()

347

In [446]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(num_df, 'chartoffset', 'label', 'value')
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'chart', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


#### Bin

In [447]:
bin_df['value'] = pd.to_numeric(bin_df['value'], errors='coerce')
bin_df = bin_df.dropna(subset=['value'])
bin_df['value'] = bin_df['value'].astype(int)
bin_df['itemid'].nunique()

161

In [448]:
bin_df = bin_df.groupby(['subject_id','label']).max().reset_index()
bin_df = bin_df.pivot_table(index='subject_id', columns='label', values='value')
bin_df.columns = ['chart_' + col for col in bin_df.columns]

In [449]:
myPredictorsDf2 = myPredictorsDf.merge(bin_df, on='subject_id', how='left')
myPredictorsDf2 = myPredictorsDf2.fillna(0)

#### Cat

In [450]:
cat_df = cat_df.dropna(subset=['value'])
cat_df['value'] = cat_df['value'].str.split(';')
cat_df = cat_df.explode('value')
cat_df['itemid'].nunique()

952

In [451]:
cat_df_agg = cat_df.groupby(['subject_id', 'label'])['value'].agg(list).reset_index()
pivoted_df = cat_df_agg.pivot(index='subject_id', columns='label', values='value').fillna('')
encoded_dict = {}
for label in pivoted_df.columns:
    mlb = MultiLabelBinarizer()
    encoded = mlb.fit_transform(pivoted_df[label])
    encoded_dict.update({f"chart_{label}_{cls}": encoded[:, i] for i, cls in enumerate(mlb.classes_)})
binary_df = pd.DataFrame(encoded_dict, index=pivoted_df.index)
binary_df.reset_index(inplace=True)

In [452]:
myPredictorsDf3 = myPredictorsDf.merge(binary_df, on='subject_id', how='left')
myPredictorsDf3 = myPredictorsDf3.fillna(0)

#### Labs

In [453]:
labs_df['value'] = pd.to_numeric(labs_df['value'], errors='coerce')
labs_df = labs_df.dropna(subset=['value'])
labs_df['itemid'].nunique()

22

In [454]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(labs_df, 'chartoffset', 'label', 'value')
myPredictorsDf4 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


### Outputevents

In [455]:
outputevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/outputevents.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [456]:
outputevents_df = pd.merge(outputevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
outputevents_df['charttime'] = pd.to_datetime(outputevents_df['charttime'],errors='coerce')
outputevents_df['time'] = pd.to_datetime(outputevents_df['time'],errors='coerce')
outputevents_df['chartoffset'] = (outputevents_df['charttime']-outputevents_df['time']).dt.total_seconds()/60
outputevents_df = outputevents_df[outputevents_df['chartoffset']>=0]
outputevents_df = outputevents_df.sort_values(by=['subject_id','chartoffset'])
outputevents_df = outputevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [457]:
columns_to_keep = ['subject_id','value','chartoffset','itemid','label','param_type']
output_df = outputevents_df[columns_to_keep]
output_df = output_df.dropna(subset='label')
output_df = output_df[output_df['chartoffset']<=myHours]

In [458]:
num_output_df = output_df[output_df['param_type']=='Numeric']
cat_output_df = output_df[output_df['param_type']=='Text']

In [459]:
num_output_df = num_output_df.drop(columns=['param_type'])
cat_output_df = cat_output_df.drop(columns=['param_type'])

#### Num

In [460]:
num_output_df['value'] = pd.to_numeric(num_output_df['value'], errors='coerce')
num_output_df = num_output_df.dropna(subset=['value'])
num_output_df['itemid'].nunique()

38

In [461]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(num_df, 'chartoffset', 'label', 'value')
myPredictorsDf5 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'output', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


#### Cat

In [462]:
cat_output_df['itemid'].nunique()

0

### Inputevents

In [463]:
inputevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/inputevents.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [464]:
inputevents_df = pd.merge(inputevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
inputevents_df['starttime'] = pd.to_datetime(inputevents_df['starttime'],errors='coerce')
inputevents_df['time'] = pd.to_datetime(inputevents_df['time'],errors='coerce')
inputevents_df['offset'] = (inputevents_df['starttime']-inputevents_df['time']).dt.total_seconds()/60
inputevents_df = inputevents_df[inputevents_df['offset']>=0]
inputevents_df = inputevents_df.sort_values(by=['subject_id','offset'])
inputevents_df = inputevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [465]:
columns_to_keep = ['subject_id','offset','itemid','label']
input_df = inputevents_df[columns_to_keep]
input_df = input_df.dropna(subset='label')
input_df = input_df[input_df['offset']<=myHours]

In [466]:
encoder = OneHotEncoder(sparse=False, dtype=int)
encoded_matrix = encoder.fit_transform(input_df[['label']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['label']))
encoded_df.rename(columns=lambda col: col.replace('label_', 'input_'), inplace=True)
encoded_df['subject_id'] = input_df['subject_id'].values
bin_input_df = encoded_df.groupby('subject_id').max().reset_index()

/home/idies/miniconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [467]:
myPredictorsDf6 = myPredictorsDf.merge(bin_input_df, on='subject_id', how='left')
myPredictorsDf6 = myPredictorsDf6.fillna(0)

### Medication Administrations

In [468]:
med_admin_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/emar.csv', myIds)

Chunk 0 Processed.
Chunk 20 Processed.
Processing finished.


In [469]:
med_admin_df = med_admin_df.dropna(subset=['medication'])

In [470]:
med_admin_df = pd.merge(med_admin_df,ca_df[['subject_id','time']],on='subject_id',how='right')
med_admin_df['charttime'] = pd.to_datetime(med_admin_df['charttime'],errors='coerce')
med_admin_df['time'] = pd.to_datetime(med_admin_df['time'],errors='coerce')
med_admin_df['chartoffset'] = (med_admin_df['charttime']-med_admin_df['time']).dt.total_seconds()/60
med_admin_df = med_admin_df[med_admin_df['chartoffset']>=0]
med_admin_df = med_admin_df.sort_values(by=['subject_id','chartoffset'])

In [471]:
columns_to_keep = ['subject_id','chartoffset','medication']
meds_df = med_admin_df[columns_to_keep]
meds_df = meds_df[meds_df['chartoffset']<=myHours]
meds_df['medication'] = meds_df['medication'].str.lower().str.replace(' ', '_')

In [472]:
encoder = OneHotEncoder(sparse=False, dtype=int)
encoded_matrix = encoder.fit_transform(meds_df[['medication']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['medication']))
encoded_df.rename(columns=lambda col: col.replace('medication_', 'med_'), inplace=True)
encoded_df['subject_id'] = meds_df['subject_id'].values
bin_meds_df = encoded_df.groupby('subject_id').max().reset_index()

/home/idies/miniconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [473]:
myPredictorsDf7 = myPredictorsDf.merge(bin_meds_df, on='subject_id', how='left')
myPredictorsDf7 = myPredictorsDf7.fillna(0)

### Other Diagnoses

In [ ]:
# def nameSearchCardiacArrest(text):
#     text = str(text).lower()
#     patterns = [r'cardia.*rrest',
#                 r'cardio.*rrest',
#                 r'circulat.*rrest',
#                 r'\basystole',
#                 r'\basystolia',
#                 r'\bpea\b|pulseless elec.*act.*']
#     if any(re.search(pattern, text) for pattern in patterns):
#         exclusion_patterns = [r'history|hx|h/o',
#                               r'before cardiac arrest',
#                               r'due to cardiac arrest',
#                               r'neonatal',
#                               r'newborn']       
#         if not any(re.search(pattern, text) for pattern in exclusion_patterns):
#             return True   
#     return False

# def icdSearchCardiacArrest(text):
#     text = str(text).lower()
#     icd10_match = re.search(r'\bi46.*\b', text)
#     icd9_match = re.search(r'\b4275\b', text)
#     return bool(icd10_match or icd9_match)

In [ ]:
# dx_id_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/d_icd_diagnoses.csv')
# diagnoses_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/diagnoses_icd.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [ ]:
# dx_id_df['icd_code'] = dx_id_df['icd_code'].str.strip()
# dx_id_df['long_title'] = dx_id_df['long_title'].str.strip()
# dx_id_df['icd_search'] = dx_id_df['icd_code'].transform(icdSearchCardiacArrest)
# dx_id_df['name_search'] = dx_id_df['long_title'].transform(nameSearchCardiacArrest)
# filtered_dx_id_df = dx_id_df[(~dx_id_df['icd_search'])&(~dx_id_df['name_search'])]

In [ ]:
# diagnoses_df['icd_code'] = diagnoses_df['icd_code'].str.strip()
# dx_df = diagnoses_df.merge(filtered_dx_id_df[['icd_code','long_title']], on='icd_code', how='inner')
# dx_df['long_title'] = dx_df['long_title'].str.lower().str.replace(' ', '_', regex=True)

In [ ]:
# encoder = OneHotEncoder(sparse=False, dtype=int)
# encoded_matrix = encoder.fit_transform(dx_df[['long_title']])
# encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['long_title']))
# encoded_df.rename(columns=lambda col: col.replace('long_title_', 'dx_'), inplace=True)
# encoded_df['subject_id'] = dx_df['subject_id'].values
# bin_dx_df = encoded_df.groupby('subject_id').max().reset_index()

/home/idies/miniconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [ ]:
# myPredictorsDf8 = myPredictorsDf.merge(bin_dx_df,on='subject_id',how='left')
# myPredictorsDf8 = myPredictorsDf8.fillna(0)

### Merge

In [ ]:
columns = ['subject_id', 'gender', 'age', 'diagnosis', 'death_at_disch']
myPredictorsDf = myPredictorsDf1
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf2, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf3, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf4, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf5, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf6, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf7, on=columns, how='left')
# myPredictorsDf = myPredictorsDf.merge(myPredictorsDf8, on=columns, how='left')

### CA Diagnosis

In [481]:
myPredictorsDf['other_underlying_condition'] = (myPredictorsDf['diagnosis'] == 'Cardiac arrest due to other underlying condition').astype(int)
myPredictorsDf['underlying_cardiac_condition'] = (myPredictorsDf['diagnosis'] == 'Cardiac arrest due to underlying cardiac condition').astype(int)
myPredictorsDf['following_other_sur'] = (myPredictorsDf['diagnosis'] == 'Postprocedural cardiac arrest following other surgery').astype(int)
myPredictorsDf['following_cardiac_sur'] = (myPredictorsDf['diagnosis'] == 'Postprocedural cardiac arrest following cardiac surgery').astype(int)
myPredictorsDf = myPredictorsDf.drop(columns=['diagnosis'])

### GCS

In [487]:
mgcs_df = chartevents_df[chartevents_df['itemid']==223901]
mgcs_df['valuenum'] = pd.to_numeric(mgcs_df['valuenum'], errors='coerce')
mgcs_df = mgcs_df.dropna(subset=['valuenum'])
mgcs_df['subject_id'].nunique()

/tmp/ipykernel_2720/3198510107.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mgcs_df['valuenum'] = pd.to_numeric(mgcs_df['valuenum'], errors='coerce')


743

In [488]:
myGcsDf = mgcs_df.sort_values(by=['subject_id','chartoffset'])
myGcsDf['valuenum'] = myGcsDf.groupby('subject_id')['valuenum'].ffill().bfill()
myGcsDf['chartoffset2'] = myGcsDf.chartoffset
myGcsDf2 = myGcsDf.groupby('subject_id').agg({'chartoffset':'min', 'chartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['subject_id', 'chartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['subject_id', 'chartoffset2'], how='inner')
myGcsDfMin['valuenum'] = myGcsDfMin['valuenum'].astype(int)
myGcsDfMax['valuenum'] = myGcsDfMax['valuenum'].astype(int)

In [489]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['subject_id', 'chartoffset', 'valuenum']], on='subject_id')
myPredictorsDf.rename(columns={'chartoffset': 'first_mGCS_time', 'valuenum': 'first_mGCS'}, inplace=True)
myPredictorsDf = myPredictorsDf.merge(myGcsDfMax[['subject_id', 'chartoffset2', 'valuenum']], on='subject_id')
myPredictorsDf.rename(columns={'chartoffset2': 'last_mGCS_time', 'valuenum': 'last_mGCS'}, inplace=True)

### Hypothermia

In [493]:
cooling_status_df = chartevents_df[chartevents_df['itemid']==225052]
cooling_status_df = cooling_status_df[cooling_status_df['chartoffset']<=1440]
cooling_status_df['on'] = (cooling_status_df['value']=='On').astype(int)

In [494]:
cooling_status_df = cooling_status_df.sort_values(['subject_id', 'chartoffset'])
cooling_status_df['on_shift'] = cooling_status_df.groupby('subject_id')['on'].shift(1, fill_value=0)
cooling_status_df['change'] = (cooling_status_df['on'] != cooling_status_df['on_shift']).astype(int)
cooling_status_df['group'] = cooling_status_df.groupby('subject_id')['change'].cumsum()
on_periods = cooling_status_df[cooling_status_df['on'] == 1]
duration_df = on_periods.groupby(['subject_id', 'group']).agg(start_offset=('chartoffset', 'min'),
                                                              end_offset=('chartoffset', 'max')).reset_index()
duration_df['duration'] = duration_df['end_offset'] - duration_df['start_offset']
longest_df = duration_df.loc[duration_df.groupby('subject_id')['duration'].idxmax()]
longest_df = longest_df[['subject_id', 'start_offset', 'end_offset', 'duration']]
hyp_df = longest_df[longest_df['duration']>=720]

In [501]:
myPredictorsDf['hypothermia'] = myPredictorsDf['subject_id'].isin(hyp_df['subject_id']).astype(int)
myPredictorsDf['hypothermia'].sum()

280

In [499]:
myPredictorsDf

,subject_id,gender,age,death_at_disch,chart_first_absolute_count_-_basos,chart_first_absolute_count_-_eos,chart_first_absolute_count_-_lymphs,chart_first_absolute_count_-_monos,chart_first_absolute_count_-_neuts,chart_first_alkaline_phosphate,...,following_cardiac_sur,first_mGCS_time,first_mGCS,last_mGCS_time,last_mGCS,first_mGCS_time,first_mGCS,last_mGCS_time,last_mGCS,hypothermia
0,10004720,1,61,1,0.03,0.00,0.58,1.29,13.53,149.0,...,0,5.0,1,6485.0,2,5.0,1,6485.0,2,1
1,10010471,0,89,1,NaN,NaN,NaN,NaN,NaN,NaN,...,0,17.0,6,6717.0,6,17.0,6,6717.0,6,0
2,10038688,1,46,1,NaN,NaN,NaN,NaN,NaN,190.0,...,0,37.0,3,23026.0,1,37.0,3,23026.0,1,1
3,10067389,1,76,1,NaN,NaN,NaN,NaN,NaN,307.0,...,0,82.0,1,18485.0,1,82.0,1,18485.0,1,0
4,10079545,0,89,1,0.04,0.01,0.65,0.70,13.04,115.0,...,0,56.0,1,928.0,1,56.0,1,928.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,19904446,0,53,1,NaN,NaN,NaN,NaN,NaN,78.0,...,0,50.0,1,2334.0,1,50.0,1,2334.0,1,1
739,19914761,1,86,1,NaN,NaN,NaN,NaN,NaN,NaN,...,0,47.0,1,47.0,1,47.0,1,47.0,1,0
740,19926342,1,87,1,NaN,NaN,NaN,NaN,NaN,101.0,...,0,353.0,4,24113.0,6,353.0,4,24113.0,6,0
741,19962126,1,66,0,0.00,0.00,0.42,0.28,13.21,58.0,...,0,33.0,5,75933.0,6,33.0,5,75933.0,6,0


### Save

In [500]:
myPredictorsDf.to_csv('MIMIC_Predictors.csv', index=False)